# 12 — Previsões cegas para validação de 2026

Este notebook gera previsões sem abrir o arquivo de consumo real de 2026. O treinamento e a seleção dos cinco componentes do ensemble terminam em 2025. O clima fica fora porque os arquivos disponíveis não permitem auditar quando cada previsão meteorológica se tornou conhecida.

Os registros de 2026 são um teste externo retrospectivo, produzido em modo cego em 2 de outubro de 2026. As origens de sexta-feira simulam a rotina semanal; não são carimbos de previsões emitidas à época. O arquivo resultante deve ser congelado antes de executar o notebook de pontuação. A quantidade informada pela nutricionista é carregada apenas como baseline operacional independente, pois sua origem difere das reservas feitas por estudantes entre 2023 e 2025.

O cenário de margem fixa de 10 refeições foi escolhido no corte temporal 2023–2024 → 2025 sob a razão analítica falta:sobra de 2:1. É cenário, não custo institucional comprovado.


Reexecução segura: se a emissão já existe, este notebook confere seu SHA-256 e exibe os dados preservados. O código de geração permanece disponível para a primeira emissão em uma nova janela.

In [1]:
from pathlib import Path
import json,hashlib
import pandas as pd
_root_emissao=Path(r'F:\PRODUTO_MESTRADO')
_arquivo_emissao=_root_emissao/'resultados/validacao_2026/previsoes_cegas_2026.csv'
_manifesto_emissao=_root_emissao/'resultados/validacao_2026/manifesto_previsoes_cegas_2026.json'
PRESERVAR_EMISSAO_EXISTENTE=_arquivo_emissao.exists()
if PRESERVAR_EMISSAO_EXISTENTE:
    _manifesto=json.loads(_manifesto_emissao.read_text(encoding='utf-8'))
    _hash=hashlib.sha256(_arquivo_emissao.read_bytes()).hexdigest()
    assert _hash in json.dumps(_manifesto), 'Integridade da emissão congelada não confirmada.'
    print('Emissão existente validada por SHA-256 e preservada. Nenhuma nova emissão produzida.')
    display(pd.read_csv(_arquivo_emissao).head())


Emissão existente validada por SHA-256 e preservada. Nenhuma nova emissão produzida.


,data,origem,data_geracao_previsao,horizonte_dias,origem_treino_ate,modelo_congelado,componentes_congelados,previsao_consumo,quantidade_pura,quantidade_margem_10,reservas_nutricionista,status_reservas_2026
0,2026-02-04,2026-01-30,2026-10-02T21:57:25,5,2025-12-31,Ensemble_top5_validacao_media,"[""CatBoost_native"", ""ExtraTrees"", ""KNN"", ""Grad...",120.874253,121,131,220.0,observado
1,2026-02-05,2026-01-30,2026-10-02T21:57:25,6,2025-12-31,Ensemble_top5_validacao_media,"[""CatBoost_native"", ""ExtraTrees"", ""KNN"", ""Grad...",115.802470,116,126,220.0,observado
2,2026-02-06,2026-01-30,2026-10-02T21:57:25,7,2025-12-31,Ensemble_top5_validacao_media,"[""CatBoost_native"", ""ExtraTrees"", ""KNN"", ""Grad...",117.213462,118,128,220.0,observado
3,2026-02-09,2026-02-06,2026-10-02T21:57:25,3,2025-12-31,Ensemble_top5_validacao_media,"[""CatBoost_native"", ""ExtraTrees"", ""KNN"", ""Grad...",119.534570,120,130,214.0,observado
4,2026-02-10,2026-02-06,2026-10-02T21:57:25,4,2025-12-31,Ensemble_top5_validacao_media,"[""CatBoost_native"", ""ExtraTrees"", ""KNN"", ""Grad...",118.381924,119,129,200.0,observado


In [2]:
if not PRESERVAR_EMISSAO_EXISTENTE:
    from pathlib import Path
    from datetime import datetime
    import hashlib
    import json
    import numpy as np
    import pandas as pd
    
    ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "gold").is_dir()), None)
    if ROOT is None:
        raise RuntimeError("Raiz do projeto não localizada.")
    AS_OF_DATE = pd.Timestamp("2026-10-02")
    LINE_ID = "consumo_cardapio_calendario"
    RESULTS = ROOT / "resultados" / "validacao_2026"
    RESULTS.mkdir(parents=True, exist_ok=True)
    FORECAST_PATH = RESULTS / "previsoes_cegas_2026.csv"
    MANIFEST_PATH = RESULTS / "manifesto_previsoes_cegas_2026.json"


## 1. Ler somente as fontes permitidas e confirmar a seleção histórica

In [3]:
if not PRESERVAR_EMISSAO_EXISTENTE:
    import sys
    SCRIPTS = ROOT / "scripts"
    if str(SCRIPTS) not in sys.path:
        sys.path.insert(0, str(SCRIPTS))
    
    from feature_engineering_temporal import build_training_feature_table, read_calendar, read_gold
    from competir_modelos_temporalmente import run_competition
    
    GOLD = read_gold()
    GOLD["data"] = pd.to_datetime(GOLD["data"], errors="raise").dt.normalize()
    assert GOLD.loc[GOLD["data"].dt.year.ge(2026), "consumo_total"].isna().all(), "A base segura contém alvo real em 2026."
    TRAIN_GOLD = GOLD.loc[GOLD["data"].dt.year.le(2025)].copy()
    assert TRAIN_GOLD["consumo_total"].notna().sum() == 503, "A contagem histórica mudou; revisar antes de continuar."
    
    FEATURES_PATH = ROOT / "data" / "experimentos" / LINE_ID / "features" / "features_supervisionadas_horizonte_7d.csv"
    TRAIN_FEATURES = pd.read_csv(FEATURES_PATH, encoding="utf-8-sig", low_memory=False)
    TRAIN_FEATURES["data_alvo"] = pd.to_datetime(TRAIN_FEATURES["data_alvo"], errors="raise").dt.normalize()
    assert TRAIN_FEATURES["data_alvo"].dt.year.max() <= 2025, "Há amostra de 2026 na tabela de treino."
    
    WEIGHTS_PATH = ROOT / "resultados" / "linhas" / LINE_ID / "04_competicao_modelos_ensembles" / "pesos_ensembles_por_fold.csv"
    weights = pd.read_csv(WEIGHTS_PATH, encoding="utf-8-sig")
    weights["fold"] = pd.to_numeric(weights["fold"], errors="raise")
    ensemble_rows = weights.loc[weights["ensemble"].eq("Ensemble_top5_validacao_media")].copy()
    last_historical_fold = int(ensemble_rows["fold"].max())
    components = ensemble_rows.loc[ensemble_rows["fold"].eq(last_historical_fold), "modelo"].drop_duplicates().tolist()
    assert len(components) == 5, f"Esperava cinco componentes selecionados; encontrei {components}."
    print("Amostras históricas com consumo:", int(TRAIN_GOLD["consumo_total"].notna().sum()))
    print("Última origem histórica usada para a seleção:", last_historical_fold)
    print("Componentes congelados:", components)


## 2. Definir alvos úteis e origens simuladas de sexta-feira

In [4]:
if not PRESERVAR_EMISSAO_EXISTENTE:
    calendar = read_calendar()
    calendar["data"] = pd.to_datetime(calendar["data"], errors="raise").dt.normalize()
    safe_future = GOLD.loc[GOLD["data"].dt.year.eq(2026)].copy()
    safe_future = safe_future.loc[safe_future["data"].le(AS_OF_DATE)]
    eligible_calendar = calendar.loc[
        calendar["data"].dt.year.eq(2026)
        & calendar["data"].le(AS_OF_DATE)
        & calendar["data"].dt.dayofweek.lt(5)
        & calendar["status_legal"].astype(str).str.casefold().eq("letivo")
    ].copy()
    target_dates = pd.DatetimeIndex(eligible_calendar["data"].sort_values().unique())
    if target_dates.empty:
        raise ValueError("Nenhum dia útil letivo de 2026 até a data de corte.")
    
    def friday_origin(target):
        target = pd.Timestamp(target).normalize()
        return target - pd.Timedelta(days=target.dayofweek + 3)
    
    target_map = pd.DataFrame({"data": target_dates})
    target_map["origem"] = target_map["data"].map(friday_origin)
    origins = sorted(target_map["origem"].unique())
    folds = pd.DataFrame({"fold": np.arange(1001, 1001 + len(origins)), "origem": origins})
    print(f"Alvos elegíveis até {AS_OF_DATE.date()}: {len(target_dates)} | origens simuladas: {len(folds)}")


## 3. Gerar e congelar as previsões sem consultar os alvos de 2026

In [5]:
if not PRESERVAR_EMISSAO_EXISTENTE:
    if FORECAST_PATH.exists() or MANIFEST_PATH.exists():
        raise FileExistsError("Já existe previsão congelada. Preserve o arquivo e gere nova versão explicitamente se necessário.")
    
    predictions, failures, audit, ensemble_weights = run_competition(
        TRAIN_FEATURES,
        GOLD,
        calendar,
        folds,
        model_names=components,
        progress_callback=lambda p: print(
            f"origem {p['fold_number']}/{p['fold_count']} | modelos válidos {p['models_ok']} | falhas {p['models_failed']}"
        ),
    )
    selected = predictions.loc[predictions["modelo"].eq("Ensemble_top5_validacao_media")].copy()
    selected["data"] = pd.to_datetime(selected["data"], errors="raise").dt.normalize()
    selected["origem"] = pd.to_datetime(selected["origem"], errors="raise").dt.normalize()
    selected = selected.loc[selected["data"].isin(target_dates)].copy()
    if selected.empty or selected["data"].duplicated().any():
        raise ValueError("Previsões ausentes ou duplicadas nos alvos elegíveis.")
    if selected["consumo_real"].notna().any():
        raise AssertionError("A base segura expôs um alvo de 2026 durante a geração.")
    
    safe_map = safe_future.set_index("data")
    selected["reservas_nutricionista"] = selected["data"].map(safe_map["reservas"])
    selected["status_reservas_2026"] = selected["data"].map(safe_map["status_reservas"])
    selected["horizonte_dias"] = (selected["data"] - selected["origem"]).dt.days
    selected["data_geracao_previsao"] = pd.Timestamp.now().isoformat(timespec="seconds")
    selected["data_corte_simulada"] = selected["origem"]
    selected["origem_treino_ate"] = "2025-12-31"
    selected["modelo_congelado"] = "Ensemble_top5_validacao_media"
    selected["componentes_congelados"] = json.dumps(components, ensure_ascii=False)
    selected["previsao_consumo"] = selected["previsao"].clip(lower=0)
    selected["quantidade_pura"] = np.ceil(selected["previsao_consumo"]).astype(int)
    selected["quantidade_margem_10"] = np.ceil(selected["previsao_consumo"] + 10).astype(int)
    selected = selected.sort_values("data").reset_index(drop=True)
    columns = [
        "data", "origem", "data_geracao_previsao", "horizonte_dias", "origem_treino_ate",
        "modelo_congelado", "componentes_congelados", "previsao_consumo", "quantidade_pura",
        "quantidade_margem_10", "reservas_nutricionista", "status_reservas_2026",
    ]
    selected[columns].to_csv(FORECAST_PATH, index=False, encoding="utf-8-sig")
    digest = hashlib.sha256(FORECAST_PATH.read_bytes()).hexdigest()
    manifest = {
        "status": "congelada_antes_da_pontuacao",
        "data_geracao": pd.Timestamp.now().isoformat(timespec="seconds"),
        "data_corte": str(AS_OF_DATE.date()),
        "alvos_treino_ate": "2025-12-31",
        "arquivo_previsoes": str(FORECAST_PATH.relative_to(ROOT)),
        "sha256_previsoes": digest,
        "n_previsoes": int(len(selected)),
        "n_datas": int(selected["data"].nunique()),
        "origens_simuladas_sextas": int(selected["origem"].nunique()),
        "componentes": components,
        "selecao": "composicao do Ensemble_top5_validacao_media no ultimo fold historico da linha-base",
        "margem": 10,
        "restricoes": [
            "Este notebook não lê data/validacao_2026_restrita/consumo_real_2026.csv.",
            "Os alvos reais de 2026 não participam de treino, seleção ou previsão.",
            "O arquivo é um hindcast cego gerado em 2026-10-02; não representa previsões emitidas nas sextas históricas.",
            "Clima excluído por ausência de proveniência temporal verificável.",
            "Reservas de 2026 são estimativas da nutricionista e baseline separado, não continuação das reservas estudantis de 2023-2025.",
        ],
    }
    MANIFEST_PATH.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")
    print(f"Previsões congeladas: {len(selected)} datas | SHA-256 {digest}")
